# Phase 19 — Continuous Research: Future Enhancements

Ideas for improving the strategy. **None are implemented yet** —
each requires its own IC test and walk-forward before promotion.

Priority ranking based on expected Sharpe improvement and implementation complexity.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':130,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})

ideas = [
    {'id': 1, 'name': 'Macro Regime Filter',
     'desc': 'Cut equity exposure in recessions (PMI, yield curve). Preserve gains in bear markets.',
     'expected_gain': 'Reduce MaxDD by 30-40%. Small Sharpe hit in bull markets.',
     'complexity': 'Medium', 'priority': 'HIGH',
     'test': 'Classify months by macro regime. Compute IC and alpha separately per regime.'},

    {'id': 2, 'name': 'Carry Overlay',
     'desc': 'Add roll yield (commodity) and yield spread (bond) as secondary signal.',
     'expected_gain': 'IC improvement for bonds/commodities. Less whipsawing in sideways markets.',
     'complexity': 'Low', 'priority': 'HIGH',
     'test': 'IC test for carry signal alone. IC test for momentum + carry combined.'},

    {'id': 3, 'name': 'Cross-Sectional Momentum (Stock Level)',
     'desc': 'Replace 16 ETFs with 100+ individual stocks. Much larger opportunity set.',
     'expected_gain': 'Higher Sharpe but requires execution infrastructure.',
     'complexity': 'High', 'priority': 'MEDIUM',
     'test': 'IC test on stock universe. Walk-forward vs ETF baseline.'},

    {'id': 4, 'name': 'Sector Rotation Filter',
     'desc': 'Weight equity ETFs by sector momentum (tech vs defensives).',
     'expected_gain': 'Better intra-equity alpha. Small impact on cross-asset strategy.',
     'complexity': 'Low', 'priority': 'MEDIUM',
     'test': 'IC of sector signal on SPY sub-ETFs (XLK, XLE, XLV, etc.).'},

    {'id': 5, 'name': 'Value Overlay',
     'desc': 'Tilt away from expensive assets (P/E, Shiller CAPE, bond yield spread).',
     'expected_gain': 'Reduce drawdown in bubble environments. Long reversion cycles.',
     'complexity': 'Medium', 'priority': 'LOW',
     'test': 'IC test for value signal on 5Y forward returns.'},

    {'id': 6, 'name': 'Defensive Cash Overlay',
     'desc': 'Move to BIL/SHY when ALL ETFs are below 200DMA.',
     'expected_gain': 'Partially implemented in 200DMA filter. Formalise cash allocation.',
     'complexity': 'Low', 'priority': 'LOW',
     'test': 'Backtest: % of months in full-cash. Cost of whipsawing.'},
]

df_ideas = pd.DataFrame(ideas)
print(df_ideas[['id','name','priority','complexity','expected_gain']].to_string(index=False))

## Research Priority Matrix

In [ ]:
priority_map   = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}
complexity_map = {'Low': 1, 'Medium': 2, 'High': 3}
impact_map     = {'HIGH': 3, 'MEDIUM': 2, 'LOW': 1}

names = [i['name'] for i in ideas]
comp  = [complexity_map[i['complexity']] for i in ideas]
pri   = [priority_map[i['priority']] for i in ideas]
colors_map = {'HIGH': '#4CAF50', 'MEDIUM': '#FF9800', 'LOW': '#F44336'}
cols  = [colors_map[i['priority']] for i in ideas]

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(comp, pri, c=cols, s=350, zorder=3, edgecolors='white', lw=1.5)
for i, name in enumerate(names):
    ax.annotate(name, (comp[i], pri[i]+0.12), ha='center', fontsize=9, fontweight='bold')

ax.set_xticks([1, 2, 3]); ax.set_xticklabels(['Low', 'Medium', 'High'])
ax.set_yticks([1, 2, 3]); ax.set_yticklabels(['Low', 'Medium', 'High'])
ax.set_xlabel('Implementation Complexity'); ax.set_ylabel('Expected Impact')
ax.set_title('Research Priority Matrix\n(Green = High Priority, Orange = Medium, Red = Low)',
             fontweight='bold')
ax.set_xlim(0.5, 3.5); ax.set_ylim(0.5, 3.7)
ax.grid(True, alpha=0.3)

from matplotlib.patches import Patch
legend_elems = [Patch(facecolor='#4CAF50', label='High Priority'),
                Patch(facecolor='#FF9800', label='Medium Priority'),
                Patch(facecolor='#F44336', label='Low Priority')]
ax.legend(handles=legend_elems, fontsize=9)
plt.tight_layout(); plt.show()

## Implementation Roadmap

In [ ]:
print('=' * 65)
print('  RESEARCH ROADMAP — ORDER OF IMPLEMENTATION')
print('=' * 65)
for idea in sorted(ideas, key=lambda x: -priority_map[x['priority']]):
    print(f'  [{idea["priority"]}] #{idea["id"]} {idea["name"]}')
    print(f'       Complexity : {idea["complexity"]}')
    print(f'       Expected   : {idea["expected_gain"]}')
    print(f'       How to test: {idea["test"]}')
    print()
print('RULE: Every idea must pass the full governance checklist')
print('      before replacing or augmenting the current strategy.')
print('=' * 65)

## Governance Gate for New Research

Before any enhancement is promoted to production:

1. **IC test**: New signal must have IC > 0 with p < 0.10
2. **Walk-forward**: Sharpe > 0.7 in >60% of out-of-sample folds
3. **Bootstrap**: 5th-pct CAGR > 0 (not luck)
4. **Comparison**: Must improve Sharpe by >0.05 vs current baseline
5. **Cost**: Net CAGR must remain positive after higher turnover costs

The current baseline: **Sharpe ≈ 1.08, CAGR ≈ 12%, MaxDD ≈ -13%**